In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import time 

In [ ]:
os.system("scripts/compile_project.sh 100 100 500 1.0 1.0 1.0 > compilation.log 2>&1")
with open("compilation.log") as f:
    print(f.read())


In [ ]:
# Add the relative path to the build directory
sys.path.insert(0, os.path.join(os.getcwd(), 'build'))

#import mlmcprotons
from mlmcprotons import MLMCprotons

### Initial Dose Distribution

In [ ]:
seed = 123456789
conversionTable = "HU2materialWater.csv"
phantom = "phantom.dat"
maxNumthreads = 8
miniStepShift = 1e-6        # Can not have integer step sizes
nPrim = int(1e4)

start = time.time()

protonSim = MLMCprotons(seed, conversionTable)
protonSim.loadPhantom(phantom, 100, 100, 500)
protonSim.simulateBeam(0, nPrim, 2+miniStepShift, 'z', 3, 0, 0.0, 1, 50, 50, 0, 200, 2.3, maxNumthreads, 0)

end = time.time()

print(f"Execution time: {end - start:.6f} seconds")

dose = protonSim.yieldDose(0).grid
doseSlice = dose[50,:,:]

# Plotting the dose slice
plt.imshow(doseSlice, cmap='hot', interpolation='nearest')
plt.title(f'MC Dose Distribution at Slice X = {50}, MLMC Replica')
plt.xlabel('Z [mm]')
plt.ylabel('Y [mm]')
plt.show()

In [ ]:

seed = 123456789
conversionTable = "HU2materialWater.csv"
phantom = "phantom.dat"
maxNumthreads = 8
miniStepShift = 1e-6        # Can not have integer step sizes
nPrim = int(1e5)

protonSim = MLMCprotons(seed, conversionTable)
protonSim.loadPhantom(phantom, 100, 100, 500)

for l in range(0, 4):
    start = time.time()
    protonSim.simulateBeam(0, nPrim, 2**(2-l)+miniStepShift, 'z', 3, 0, 0, 1, 50, 50, 0, 230, 2.3, maxNumthreads, 0) 
    end = time.time()
    print(f"Execution time for level {l}: {end - start:.6f} seconds")

fig, ax = plt.subplots(figsize=(8, 5))
for l in range(0, 4):
    dose = protonSim.yieldDose(l).grid
    integralDose = np.sum(dose, axis=(0, 1))
    integralDose /= np.max(integralDose)
    ax.plot(integralDose, label=f'Initial step {2**(2-l)}mm')
ax.set_xlabel('Depth [mm]')
ax.set_ylabel('Dose [%]')
ax.set_title('Integral Dose vs Depth')
ax.legend()
plt.show()

# Execution time for level 0: 2.796716 seconds
# Execution time for level 1: 3.885104 seconds
# Execution time for level 2: 8.051540 seconds
# Execution time for level 3: 13.364765 seconds


### Variance at Coursest Level

In [ ]:
nDosesCoarsest = 5
protonSim = MLMCprotons(seed, conversionTable)
protonSim.loadPhantom(phantom, 100, 100, 500)
dosesCoarsest = np.zeros((100, 100, 500, nDosesCoarsest))
finalDose = np.zeros((100, 100, 500))
nPrim = 10000
for n in range(nDosesCoarsest):
    protonSim.simulateBeam(
        0, nPrim, 1 + miniStepShift, 'z',
        3, 0, 0, 1,
        50, 50, 0,
        230, 2.3,
        maxNumthreads, 0
    )
    dosesCoarsest[:,:,:,n] = protonSim.yieldDose(n).grid/nPrim
    finalDose += dosesCoarsest[:,:,:,n]
variance = np.sum(np.var(dosesCoarsest, axis=3))

print(variance)

In [ ]:
seed = 123456781
conversionTable = "HU2materialWater.csv"
phantom = "phantom.dat"
maxNumthreads = 8
miniStepShift = 1e-6  # Can not have integer step sizes
nPrim = int(1e4)

protonSim = MLMCprotons(seed, conversionTable)
protonSim.loadPhantom(phantom, 100, 100, 500)



protonSim.simulateBeamWithShadow(
    0, nPrim, 1.0+miniStepShift, 'z',
    3, 0, 0, 1,
    50, 50, 0,
    230, 2.3, 
    maxNumthreads, 1, True
)


doseDifference = protonSim.yieldDose(0).grid/nPrim
doseVariance = protonSim.yieldDose(1).grid/nPrim

fig, ax = plt.subplots(nrows=2, ncols=1, figsize=(10, 8))

# Plot dose difference slice
im0 = ax[0].imshow(doseDifference[50,:,:], cmap='viridis', origin='lower')
ax[0].set_title('Expected Dose Difference at Slice X=50')
ax[0].set_xlabel('Z [mm]')
ax[0].set_ylabel('Y [mm]')
fig.colorbar(im0, ax=ax[0], orientation='vertical', label='Dose Difference')

# Plot dose variance slice
im1 = ax[1].imshow(doseVariance[50,:,:], cmap='inferno', origin='lower')
ax[1].set_title(f'Dose Variance at Slice X=50, Variance Sum = {np.sum(doseVariance):.4e}')
ax[1].set_xlabel('Z  [mm]')
ax[1].set_ylabel('Y [mm]')
fig.colorbar(im1, ax=ax[1], orientation='vertical', label='Dose Variance')

plt.tight_layout()
plt.show()

### Compare Variance across different Levels

In [ ]:
seed = 123456789
conversionTable = "HU2materialWater.csv"
phantom = "phantom.dat"
maxNumthreads = 8
miniStepShift = 1e-6  # Can not have integer step sizes
nPrim = int(1e4)

varianceSums = []
protonSim = MLMCprotons(seed, conversionTable)
protonSim.loadPhantom(phantom, 100, 100, 500)

for l in range(1, 6):

    start = time.time()

    protonSim.simulateBeamWithShadow(
        0, nPrim, 2**(1-l) + miniStepShift, 'z',
        3, 0, 0, 1,
        50, 50, 0,
        230, 2.3,
        maxNumthreads, 1, True
    )

    doseVar = protonSim.yieldDose(2*l-1).grid
    scalarVar = np.sum(doseVar)/nPrim
    varianceSums.append(scalarVar)

    end = time.time()
    print(f"Var at level {l} = {scalarVar}")
    print(f"Calculations at level {l} finnished in {end-start:.2f} seconds")

In [ ]:
levels = range(1, 6)
log_variance = np.log(varianceSums)

plt.scatter(levels, log_variance)

# Add annotations with formatted values
for l, val in zip(levels, varianceSums):
    plt.annotate(f'{val:.4e}',  # format with 4 decimal scientific notation
                 (l, np.log(val)),
                 textcoords="offset points", 
                 xytext=(0, 8),  # offset above the point
                 ha='center',
                 fontsize=9,
                 color='blue')

plt.xlabel('Level (l)')
plt.ylabel('Log of Variance Sum')
plt.title('Variance Sum vs Level')
plt.grid(True)
plt.show()

### Calculate Expected Difference and Variance per Level

In [ ]:
nDosesCoarsest = 5

def calculateLevel(l, nPrim, finalDose):
    protonSim = MLMCprotons(seed, conversionTable)
    protonSim.loadPhantom(phantom, 100, 100, 500)
    
    # Calculate variance of coarsest layer with respect to batches of the layer it self
    if l == 0: 
        dosesCoarsest = np.zeros((100, 100, 500, nDosesCoarsest))
        for n in range(nDosesCoarsest):
            nPrimBatch = int(nPrim/nDosesCoarsest)
            protonSim.simulateBeam(
                0, nPrimBatch, 2**(1-l) + miniStepShift, 'z',
                3, 0, 0, 1,
                50, 50, 0,
                230, 2.3,
                maxNumthreads, 0
            )
            dosesCoarsest[:,:,:,n] = protonSim.yieldDose(n).grid/nPrimBatch
            finalDose += dosesCoarsest[:,:,:,n]
        variance = np.sum(np.var(dosesCoarsest, axis=3))
    else:
        protonSim.simulateBeamWithShadow(
            0, nPrim, 2**(1-l) + miniStepShift, 'z',
            3, 0, 0, 1,
            50, 50, 0,
            230, 2.3,
            maxNumthreads, 1, True
        )
        finalDose += protonSim.yieldDose(0).grid/nPrim
        varianceGrid = protonSim.yieldDose(1).grid
        variance = np.sum(varianceGrid)/nPrim
        
    return variance

### MLMC Algorithm (Update Manually)

In [ ]:

err2 = 1e-6
totalVar = np.inf
maxL = 5

L = 4  # current number of active levels
# First iteration [10000, 5000, 2500]
# Second iteration [27060, 3306, 872, 242] 
nSamplesPerLevel = [27060, 3306, 872, 242] # + [0] * (maxL - 2)

finalDose = np.zeros((100, 100, 500))
varAcrossLevels = [0.0 for _ in range(maxL)]

start = time.time()

# Simulate across all current active levels
for l, nPrim in enumerate(nSamplesPerLevel):

    varAcrossLevels[l] = calculateLevel(l, nPrim, finalDose)
        
    print(f"Variance at level {l} = {varAcrossLevels[l]}")

end = time.time()

print(f"Calculating samples took {end-start:.3f} seconds")





In [ ]:
# Compute total variance based on current estimates
totalVar = sum([varAcrossLevels[l] / nSamplesPerLevel[l] for l in range(L)])

# Compute optimal number of samples per level (based on cost ~ 2^l)
costPerSample = [2**l for l in range(L)]
sqrtTerm = [np.sqrt(varAcrossLevels[l] * costPerSample[l]) for l in range(L)]
sumSqrt = sum(sqrtTerm)

optimalNSamples = [
    int(np.ceil(1/err2 * sumSqrt * np.sqrt(varAcrossLevels[l] / costPerSample[l])))
    for l in range(L)
]

print(f"total var: {totalVar}")
print(f"optimal samples: {optimalNSamples}")

# total var: 2.0255958294704723e-06
# optimal samples: [27060, 3306, 872, 242]

### MLMC Algorithm (Automatic)

In [ ]:

err2 = 1e-6
totalVar = np.inf
maxL = 6
L = 2  # current number of active levels
computedSamplesPerLevel = False

finalDose = np.zeros((100, 100, 500))
nSamplesPerLevel = [10000, 5000] + [0] * (maxL - L)


while totalVar > err2:

    finalDose = np.zeros((100, 100, 500))
    varAcrossLevels = [0.0 for _ in range(maxL)]

    protonSim = MLMCprotons(seed, conversionTable)
    protonSim.loadPhantom(phantom, 100, 100, 500)

    start = time.time()

    # Simulate across all current active levels
    for l, nPrim in enumerate(nSamplesPerLevel[:L]):
        if nPrim == 0:
            continue

        varAcrossLevels[l] = calculateLevel(l, nPrim, finalDose)

        print(f"Variance at level {l} = {varAcrossLevels[l]}")

    end = time.time()

    print(f"Calculating samples took {end-start:.3f} seconds")

    # Compute total variance based on current estimates
    totalVar = sum([varAcrossLevels[l] / nSamplesPerLevel[l] for l in range(L)])

    # Compute optimal number of samples per level (based on cost ~ 2^l)
    costPerSample = [2**l for l in range(L)]
    sqrtTerm = [np.sqrt(varAcrossLevels[l] * costPerSample[l]) for l in range(L)]
    sumSqrt = sum(sqrtTerm)

    optimalNSamples = [
        int(np.ceil(1/err2 * sumSqrt * np.sqrt(varAcrossLevels[l] / costPerSample[l])))
        for l in range(L)
    ]

    print(f"total var: {totalVar}")
    print(f"optimal samples: {optimalNSamples}")

    # Update samples per level if needed
    updated = False
    for l in range(L):
        if optimalNSamples[l] > nSamplesPerLevel[l]:
            nSamplesPerLevel[l] = optimalNSamples[l]
            updated = True

    # If all variances on current levels are small and adding a level would help, add new level
    if not updated and L < maxL:
        L += 1
        nSamplesPerLevel[L-1] = 500  # start with a small number of samples on the new level


Variance at level 0 = 0.018731116066848175
Variance at level 1 = 0.000559169624466449
Variance at level 2 = 7.77430905145593e-05
Variance at level 3 = 1.1941318916797172e-05
Calculating samples took 82.518 seconds
total var: 2.025595825714065e-06
optimal samples: [270592, 33059, 8717, 2416]
Variance at level 0 = 0.0007085717417582916
Variance at level 1 = 0.0005557904369197786
Variance at level 2 = 7.742534944554791e-05
Variance at level 3 = 1.1710644685081206e-05
Calculating samples took 1360.218 seconds
total var: 3.315990552612034e-08
optimal samples: [23222, 14543, 3839, 1056]

In [ ]:

plt.imshow(finalDose[50,:,:], cmap='hot', interpolation='nearest')
plt.title(f'MLMC Dose Distribution at Slice X = {50}')
plt.xlabel('Z [mm]')
plt.ylabel('Y [mm]')
plt.show()

In [ ]:
plt.imshow(
    doseSlice[25:75, 300:400],
    cmap='hot',
    interpolation='nearest',
    extent=[300, 400, 75, 25]  # extent = [x_min, x_max, y_min, y_max]
)
plt.title('MC Dose Distribution at Slice X = 50 (MLMC Replica)')
plt.xlabel('Z [mm]')
plt.ylabel('Y [mm]')
plt.show()

In [ ]:

plt.imshow(
    finalDose[50,25:75,300:400],
    cmap='hot',
    interpolation='nearest',
    extent=[300, 400, 75, 25]  # extent = [x_min, x_max, y_min, y_max]
)
plt.title('MLMC Dose Distribution at Slice X = 50')
plt.xlabel('Z [mm]')
plt.ylabel('Y [mm]')
plt.show()